# Format 1 — The Agent Arena
## One agent, many environments

**Gemini's prediction: 60% probability this is Round 2.**

Meta's strategic bet: OpenEnv replaces Gymnasium as the standard for agentic evaluation. To prove it, they drop 5–10 **unseen** environments on Day 1 and ask: can one agent solve them all?

This notebook teaches you the **universal agent pattern** — an `AgentRouter` that reads an OpenEnv Pydantic schema and dispatches to the right sub-policy.

### What you'll learn
1. **Action space taxonomy** — discrete, continuous, structured/text
2. **Schema introspection** — reading a Pydantic model to figure out how to act
3. **Constrained decoding** — forcing an LLM to output valid actions
4. **AgentRouter** — the glue that ties it together
5. **Arena benchmarking** — one agent, three envs, one score

The core mental shift: **stop thinking "my agent solves GridOps." Start thinking "my agent solves any OpenEnv you give it."**

---
## Part 1 — The action space taxonomy

Every RL environment has an action space. In an arena, you'll see three types:

| Type | Example | Natural agent |
|---|---|---|
| **Discrete** | "email bucket ∈ {spam, inbox, priority}" | LLM with logit constraints, or Q-network |
| **Continuous** | "battery_dispatch ∈ [-1, 1]" | Gaussian policy (PPO) or LLM with parsed float |
| **Structured** | "{'tool': 'search', 'args': {...}}" | LLM with constrained JSON decoding |

A winning arena agent doesn't hard-code for any one type. It **inspects** the env's Pydantic schema and chooses the right strategy.

### Why Pydantic schemas matter

OpenEnv requires every Action to be a Pydantic `BaseModel`. That means every field has:
- A name
- A type annotation (`float`, `int`, `str`, `Literal[...]`)
- Optional constraints (`ge=0.0, le=1.0`)
- An optional description

This is **machine-readable metadata**. Your agent can read it at runtime.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('.'))

from gridops.models import GridOpsAction

# Pydantic exposes the schema as JSON
schema = GridOpsAction.model_json_schema()
print(json.dumps(schema, indent=2))

# Notice: each field has type, range (minimum/maximum), description.
# An agent can read this and know: 3 continuous actions in specific ranges.

---
## Part 2 — Building a mini-arena

We'll build three toy envs with very different action types. Each has the OpenEnv-ish interface (`reset`, `step`, a Pydantic Action model). This is what Meta's arena will look like — a bag of heterogeneous envs.

### Env A — Email Triage (Discrete action)
Given a subject line, classify into one of 3 buckets. Each correct classification +1 reward.

### Env B — Dial Control (Continuous action)
Two dials, set them to match a hidden target. Reward = −‖dials − target‖².

### Env C — Tool Call (Structured action)
Output a JSON with a `tool` name and `args`. Reward +1 if correct tool picked for the query.

In [ ]:
import numpy as np
from typing import Literal
from pydantic import BaseModel, Field

# ──────────────── Env A — Email Triage (Discrete) ────────────────
class EmailAction(BaseModel):
    bucket: Literal['spam', 'inbox', 'priority'] = Field(description='Which bucket')

class EmailObs(BaseModel):
    subject: str
    correct: str  # (for scoring — agent doesn't see this in reality)

class EmailEnv:
    ACTION_MODEL = EmailAction
    SUBJECTS = [
        ('You won $1M claim now!!!', 'spam'),
        ('Q3 report review meeting', 'inbox'),
        ('URGENT: Server is on fire', 'priority'),
        ('Viagra sale 90% off', 'spam'),
        ('Lunch tomorrow?', 'inbox'),
    ]
    def reset(self, seed=0):
        self.rng = np.random.default_rng(seed); self.t = 0; self.score = 0
        self._sample()
        return EmailObs(subject=self.subject, correct='?')
    def _sample(self):
        self.subject, self._target = self.SUBJECTS[self.rng.integers(len(self.SUBJECTS))]
    def step(self, a: EmailAction):
        r = 1.0 if a.bucket == self._target else 0.0
        self.score += r; self.t += 1
        done = self.t >= 10
        if not done: self._sample()
        return EmailObs(subject=self.subject, correct='?'), r, done

# ──────────────── Env B — Dial Control (Continuous) ────────────────
class DialAction(BaseModel):
    dial_1: float = Field(ge=-1.0, le=1.0)
    dial_2: float = Field(ge=-1.0, le=1.0)

class DialObs(BaseModel):
    target_hint_1: float  # noisy hint about target
    target_hint_2: float

class DialEnv:
    ACTION_MODEL = DialAction
    def reset(self, seed=0):
        self.rng = np.random.default_rng(seed); self.t = 0
        self._target = self.rng.uniform(-1, 1, size=2)
        return DialObs(target_hint_1=self._target[0] + self.rng.normal(0, 0.1),
                       target_hint_2=self._target[1] + self.rng.normal(0, 0.1))
    def step(self, a: DialAction):
        err = np.array([a.dial_1, a.dial_2]) - self._target
        r = -float(np.sum(err**2))
        self.t += 1; done = self.t >= 5
        self._target = self.rng.uniform(-1, 1, size=2)
        return DialObs(target_hint_1=self._target[0] + self.rng.normal(0, 0.1),
                       target_hint_2=self._target[1] + self.rng.normal(0, 0.1)), r, done

# ──────────────── Env C — Tool Call (Structured) ────────────────
class ToolAction(BaseModel):
    tool: Literal['search', 'calculator', 'weather', 'translate']
    query: str = Field(default='')

class ToolObs(BaseModel):
    user_query: str

class ToolEnv:
    ACTION_MODEL = ToolAction
    QUERIES = [
        ('What is 23 * 47?', 'calculator'),
        ('Rain tomorrow in Bangalore?', 'weather'),
        ('Who won the 2024 world cup?', 'search'),
        ('"Hello" in Spanish', 'translate'),
    ]
    def reset(self, seed=0):
        self.rng = np.random.default_rng(seed); self.t = 0
        self._pick()
        return ToolObs(user_query=self.q)
    def _pick(self):
        self.q, self._target = self.QUERIES[self.rng.integers(len(self.QUERIES))]
    def step(self, a: ToolAction):
        r = 1.0 if a.tool == self._target else 0.0
        self.t += 1; done = self.t >= 8
        if not done: self._pick()
        return ToolObs(user_query=self.q), r, done

print('Three envs ready. Each exposes ACTION_MODEL (a Pydantic class).')
for E in [EmailEnv, DialEnv, ToolEnv]:
    print(f'  {E.__name__}: {E.ACTION_MODEL.__name__} fields = {list(E.ACTION_MODEL.model_fields.keys())}')

---
## Part 3 — Schema introspection: how the agent "sees" an unknown env

The key insight: **you don't need to know the env beforehand**. Pydantic tells you everything.

For each field we extract:
- **Type** (`float` vs `int` vs `str` vs `Literal`)
- **Range** (`ge`, `le` for numbers; enum values for `Literal`)
- **Description** (optional — but great for LLM prompts)

From these three facts you can classify the action space:

In [ ]:
from typing import get_args, get_origin

def classify_action_space(action_model: type[BaseModel]) -> dict:
    """Return a per-field description that the agent can reason over."""
    info = {}
    for name, field in action_model.model_fields.items():
        ann = field.annotation
        meta = {'type': 'unknown'}
        if get_origin(ann) is Literal or hasattr(ann, '__args__') and type(None) not in get_args(ann) and all(isinstance(a, str) for a in get_args(ann) or []):
            meta = {'type': 'discrete', 'choices': list(get_args(ann))}
        elif ann in (float, int):
            # Get numeric bounds from metadata
            lo, hi = None, None
            for m in (field.metadata or []):
                if hasattr(m, 'ge'): lo = m.ge
                if hasattr(m, 'le'): hi = m.le
            meta = {'type': 'continuous' if ann is float else 'int', 'range': (lo, hi)}
        elif ann is str:
            meta = {'type': 'text'}
        info[name] = meta
    # Overall class: if any field is structured/text, call it "structured"
    types = {v['type'] for v in info.values()}
    if 'text' in types or len(types) > 1:
        overall = 'structured'
    elif 'discrete' in types:
        overall = 'discrete'
    else:
        overall = 'continuous'
    return {'overall': overall, 'fields': info}

for E in [EmailEnv, DialEnv, ToolEnv, ]:
    print(E.__name__, '→', classify_action_space(E.ACTION_MODEL))

# Try on your real GridOps env
print('GridOpsAction →', classify_action_space(GridOpsAction))

---
## Part 4 — The AgentRouter

Now the universal agent. It reads the schema once, picks a strategy, and acts.

### Strategy table

| Action space | Default strategy |
|---|---|
| Discrete | LLM with `Literal`-constrained output, or random-uniform as fallback |
| Continuous | Random in range, or PPO if we trained one |
| Structured | LLM with JSON-mode + Pydantic validation |

In a real hackathon, you'd upgrade each branch over the 48 hours. But the **router never changes** — it just dispatches.

In [ ]:
class AgentRouter:
    """Universal agent — reads Pydantic schema, dispatches to sub-policy."""
    
    def __init__(self, action_model: type[BaseModel], llm_fn=None, rng=None):
        self.action_model = action_model
        self.schema_info = classify_action_space(action_model)
        self.llm_fn = llm_fn   # optional LLM callable
        self.rng = rng or np.random.default_rng(0)
    
    def __call__(self, obs) -> BaseModel:
        overall = self.schema_info['overall']
        if overall == 'discrete':
            return self._act_discrete(obs)
        elif overall == 'continuous':
            return self._act_continuous(obs)
        else:
            return self._act_structured(obs)
    
    # ── strategies (each one upgradeable independently) ──
    def _act_discrete(self, obs):
        # LLM if available, else random over the allowed choices
        first_field = next(iter(self.schema_info['fields']))
        choices = self.schema_info['fields'][first_field]['choices']
        if self.llm_fn:
            pick = self.llm_fn(obs, choices, self.action_model)
            return self.action_model(**{first_field: pick})
        return self.action_model(**{first_field: self.rng.choice(choices)})
    
    def _act_continuous(self, obs):
        kwargs = {}
        for name, meta in self.schema_info['fields'].items():
            lo, hi = meta.get('range', (-1.0, 1.0))
            lo = lo if lo is not None else -1.0
            hi = hi if hi is not None else 1.0
            kwargs[name] = float(self.rng.uniform(lo, hi))
        return self.action_model(**kwargs)
    
    def _act_structured(self, obs):
        if self.llm_fn:
            return self.llm_fn(obs, None, self.action_model)
        # fallback: pick default-like values
        kwargs = {}
        for name, meta in self.schema_info['fields'].items():
            if meta['type'] == 'discrete':
                kwargs[name] = self.rng.choice(meta['choices'])
            elif meta['type'] == 'text':
                kwargs[name] = ''
            else:
                kwargs[name] = 0.0
        return self.action_model(**kwargs)

# Smoke test: dispatch across all 3 envs
for E in [EmailEnv, DialEnv, ToolEnv]:
    env = E()
    obs = env.reset(seed=1)
    agent = AgentRouter(E.ACTION_MODEL)
    a = agent(obs)
    print(f'{E.__name__:10s} → overall={agent.schema_info["overall"]:12s} action={a}')

---
## Part 5 — Constrained decoding (the LLM backbone)

The router works, but random-over-choices is dumb. To win, the LLM should actually reason.

**The problem:** LLMs output text. You need structured data matching a Pydantic schema. Parsing failures ruin your run (Gemini's "hour 46 crash" warning).

**Three solutions, ordered by quality:**

| Library | How it works | Robustness |
|---|---|---|
| **Outlines** | Constrains token logits so output **must** match a regex/grammar/JSON schema | Bulletproof — 0% failure |
| **Instructor** | Wraps OpenAI client, uses JSON mode + Pydantic validation + retries | ~99% — API-dependent |
| Regex + Pydantic | Ask nicely in prompt, validate output | ~95% — brittle |

For the hackathon: **use Instructor** if you're calling an API (HF router, OpenAI, Anthropic). **Use Outlines** if running Llama locally via vLLM.

Below is a minimal Instructor-style wrapper (no external library) that demonstrates the core pattern — ask the LLM, validate, retry on failure.

In [ ]:
from pydantic import ValidationError
from openai import OpenAI

client = OpenAI(
    base_url=os.getenv('API_BASE_URL', 'https://router.huggingface.co/v1'),
    api_key=os.getenv('HF_TOKEN', 'dummy'),
)
MODEL = os.getenv('MODEL_NAME', 'meta-llama/Llama-3.3-70B-Instruct')

def llm_call(obs, choices, action_model: type[BaseModel], max_retries=2) -> BaseModel:
    """Prompt the LLM, parse response, validate via Pydantic. Retry on failure."""
    schema = action_model.model_json_schema()
    schema_text = json.dumps(schema, indent=2)
    obs_text = json.dumps(obs.model_dump() if hasattr(obs, 'model_dump') else obs)
    
    system = f'You must output JSON matching this schema:\n{schema_text}\nReturn ONLY the JSON.'
    user = f'Observation: {obs_text}\nOutput JSON action now.'
    
    for attempt in range(max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{'role': 'system', 'content': system}, {'role': 'user', 'content': user}],
                temperature=0.1, max_tokens=200,
            )
            text = resp.choices[0].message.content
            # extract first JSON block
            s, e = text.find('{'), text.rfind('}') + 1
            data = json.loads(text[s:e])
            return action_model(**data)   # Pydantic validates
        except (json.JSONDecodeError, ValidationError, KeyError) as ex:
            if attempt == max_retries:
                # Final fallback — whatever the AgentRouter's non-LLM branch would do
                return action_model.model_construct()  # bypasses validation
            user += f'\n\nYour last attempt failed: {ex}. Try again, JSON only.'
    
# (We don't actually call the API here — just show the pattern is sound.)
print('llm_call defined. Pass it as AgentRouter(model, llm_fn=llm_call).')

---
## Part 6 — Arena benchmark

Run the same router against all three envs. A **normalized arena score** treats each env's best possible as 1.0.

This is the kind of plot that wins arena hackathons — one agent, one bar per env, all above the "random" baseline.

In [ ]:
def run_one(env_cls, agent_fn, seed=0, episodes=5):
    totals = []
    for ep in range(episodes):
        env = env_cls(); obs = env.reset(seed=seed+ep); r_sum = 0.0
        while True:
            a = agent_fn(obs)
            obs, r, done = env.step(a); r_sum += r
            if done: break
        totals.append(r_sum)
    return np.mean(totals)

def random_agent_factory(E):
    return AgentRouter(E.ACTION_MODEL)  # no LLM → falls back to random

# Expected best: Email 10, Dial 0 (0 error), Tool 8
BEST = {'EmailEnv': 10.0, 'DialEnv': 0.0, 'ToolEnv': 8.0}
WORST = {'EmailEnv': 0.0, 'DialEnv': -20.0, 'ToolEnv': 0.0}

results = {}
for E in [EmailEnv, DialEnv, ToolEnv]:
    score = run_one(E, random_agent_factory(E))
    # normalize to 0..1
    norm = (score - WORST[E.__name__]) / (BEST[E.__name__] - WORST[E.__name__] + 1e-9)
    results[E.__name__] = {'raw': score, 'norm': norm}

print(f'{"Env":15s} {"Raw":>10s} {"Normalized":>12s}')
for name, r in results.items():
    print(f'{name:15s} {r["raw"]:10.2f} {r["norm"]:12.2f}')
print(f'\nArena score (avg normalized): {np.mean([r["norm"] for r in results.values()]):.3f}')
print('\nThis is your baseline. Every LLM/RL upgrade should push this up.')

---
## Part 7 — Hackathon application

If Gemini is right and Round 2 is an arena, **this pattern IS your submission**:

```
/hackathon
├── router.py            # AgentRouter (this notebook)
├── classifiers.py       # classify_action_space + obs introspection
├── llm_backend.py       # Instructor/Outlines-wrapped LLM
├── rl_backend.py        # small PPO that auto-adapts to any continuous env
├── strategies/
│   ├── discrete.py      # LLM-with-choices or tabular Q-learning
│   ├── continuous.py    # PPO with dynamic obs normalization
│   └── structured.py    # LLM with JSON + retry
└── eval.py              # arena benchmark runner
```

### The 48-hour execution plan

| Hours | Deliverable |
|---|---|
| 0–2 | Introspect every env they give you. Print action schemas. Write down which strategy each needs. |
| 2–8 | Random baseline on all envs. Numeric scores. |
| 8–18 | LLM strategy on discrete + structured. Instructor integration. |
| 18–30 | PPO on continuous envs. Run in parallel across envs. |
| 30–42 | Iterate on weakest env — that drags the arena score the most |
| 42–48 | Polish demo, rehearse pitch |

### The demo visual
One bar chart: **X-axis = envs, Y-axis = normalized score, two bars per env (Random baseline vs AgentRouter)**. If your router clears random on 8/10 envs, you win the format.

### The judges' hook
> "I built one agent that reads any OpenEnv Pydantic schema and adapts. Here it is scoring on 10 envs it has never seen before."

That sentence lands. The code to back it up is this notebook, productionized.